# T128 saved-checkpoint submissions

Inference only for original run `11-04-01-58_E_pyramid-ver-33-T128-asam-t122reg` (original E33). Two saved checkpoints, best848 and last900; raw/EMA × plain/flip yields eight CSVs. Reads checkpoints on Drive, writes new non-checkpoint exports in the original test run folder. No training function or training call is included. Original launch notebook and completed outputs remain unchanged.


In [18]:
# LAUNCH COPY T128 (registered 2026-11-03, owner J.H.): ASAM control = T122 + ASAM900. Identical to T127 except
# weight decay 5e-5 and label smoothing 0 (T122's values); bottleneck PyramidNet-110 alpha 128, 4,948,490 parameters,
# model fingerprint = T122. Only the lines marked "T128 launch copy" differ from E_pyramid_t106_shakedrop.ipynb;
# the pre-declared comparison (vs T122 / T127) is in checklist T128.
# Version: E_VERSION_FLOOR 32, so the run takes E >= 33 on J.H.'s Drive (one above T127, so two simultaneous launches
# cannot share a version); check the printed "Run ID:" line.
# COMPILE "reduce-overhead" = the SAM CUDA-graph path; before any run folder exists the automatic launch preflight
# compares it with the reference "default" on this GPU ("[SAM preflight] ... passed"). If the preflight raises, set
# COMPILE = "default" (same update, slower) and Run all again. Measured on a Colab L4 (2026-11-04, bf16): bottleneck-110
# 333.9 ms/step reduce-overhead vs 346.8 default (~128 s/epoch, ~32 h per 900 epochs); A100 not yet measured
# (expected ~55-60 s/epoch, ~14-15 h). Long run: resume after a Colab disconnect with
# RESUME_RUN = "<printed run ID>", RUN_NAME = None, nothing else changed (COMPILE may switch on resume).
# Scheme E (model E, artifact stem E_pyramid): PyramidNet with optional ShakeDrop and bottleneck blocks; section 4.
# Saved preset T120 = the T106 recipe with three changes: DROP_PATH 0.1 -> SHAKEDROP 0.5, CROP_PADDING_MODE
# "constant" -> "reflect" and WEIGHT_DECAY 5e-4 -> 1e-4; basic blocks, depth 110, alpha 98 (4,963,991 parameters),
# cosine schedule. T121 changes only BLOCK = "bottleneck", DEPTH = 164, ALPHA = 100 (4,924,834 parameters), TASK_ID
# and RUN_NAME. E-only training switches SAM (with SAM_ETA / SAM_M), CUTMIX and MIXSTYLE are off.
# Export final/EMA/flip (CHECKPOINT="last", WEIGHTS="ema", TTA=True) like T106; best is also evaluated.
# Edit the values below, then Run all.

RUN_NAME = "asam-t122reg"  # T128 launch copy (was: "shakedrop-reflect")
RESUME_RUN = None        # exact saved run ID to resume, e.g. "10-30-08-40_A_resnet-ver-1-T00-cutout-ema"
EPOCHS = 900  # T128 launch copy (was: 300)
                         # halves the epochs for equal compute (ASAM paper: SGD 400 vs SAM/ASAM 200; PyramidNet-272 1800
                         # vs 900), so SAM at EPOCHS = 150 costs about one 300-epoch SGD run. Not changed automatically.
AUG = "trivialaug"       # T106: basic crop/flip + TrivialAugmentWide (per image on the CPU) + AUTOAUG_ERASE
                         # | "basic" | "cutout" | "autoaug": basic + CIFAR10 AutoAugment + AUTOAUG_ERASE
SEED = 0                 # model initialization, data order and augmentation draws
VAL_SIZE = 5000          # class-balanced validation hold-out (multiple of 10); 0 = train on all 50,000 images
EMA = True               # also keep (and evaluate) an exponential moving average of the weights
AMP = "bf16"             # CUDA only: "auto" = bf16 on GPUs with native bf16 (e.g. A100), else fp16 | "off" | "bf16" | "fp16"
CKPT_EVERY = None        # write last-ckpt.pt every N epochs; None = 5 on Colab (Drive writes are slow), else 1
TRAIN_SUBSET = 0         # > 0: train on this many images only (quick tests; multiple of 10)
# New runs never overwrite previous outputs. Set RESUME_RUN explicitly to continue a run.

# Optimization
BATCH_SIZE = 128         # kept with LR 0.1: the same LR/batch ratio as Fast AutoAugment's PyramidNet-272 (64 / 0.05)
LR = 0.1                 # and the ShakeDrop paper (128 / 0.1); the ASAM paper used 256 / 0.05
MOMENTUM = 0.9
NESTEROV = True
WEIGHT_DECAY = 5e-5  # T128 launch copy (was: 1e-4)
                         # PyramidNet-272 + ShakeDrop config uses 5e-5. With ShakeDrop + TrivialAugment + Cutout the
                         # stronger 5e-4 may over-regularize. Applies to all parameters unless WD_SKIP_BN_BIAS
WD_SKIP_BN_BIAS = False  # True: no weight decay on BN parameters and biases
WARMUP_EPOCHS = 5
LR_SCHEDULE = "cosine"   # "cosine" (T106: linear warmup, then cosine to 0) | "multistep" (same warmup, then the LR is
                         # multiplied by LR_GAMMA after each milestone epoch; A/C T66/T68 semantics). Measured 300-epoch
                         # pairs: A step 96.92% vs cosine 96.86% (T66 vs T65), C step 96.96% vs cosine 96.92% (T68 vs
                         # T67), within noise. The literature mostly uses cosine (Fast AutoAugment PyramidNet-272: cosine
                         # + 5 warmup epochs); the original ShakeDrop paper stepped at epochs 150 and 225 of 300
LR_MILESTONE_FRACTIONS = (0.5, 0.75)  # multistep only: milestones as fractions of EPOCHS (150 and 225 of 300), saved
                         # with the resolved epochs (rounded half up); ignored for cosine
LR_GAMMA = 0.1           # multistep only: LR factor at each milestone
LABEL_SMOOTHING = 0.0  # T128 launch copy (was: 0.1)
EMA_DECAY = 0.999
# Sharpness-aware minimization around the SGD above (section 5); off in this preset. About 2x s/epoch (see EPOCHS).
# The code follows github.com/davda54/sam: e = rho * T^2 g / (||T g|| + 1e-12), all parameters (BN and bias included)
# perturbed, weight decay only in SGD. With SHAKEDROP > 0 every second pass replays its first pass's ShakeDrop draws.
SAM = True  # T128 launch copy (was: False)
SAM_RHO = 1.0  # T128 launch copy (was: 0.05)
                         # teammate T112 used 2.0). The ASAM paper used 0.5 (CIFAR-10 models) and 1.0 (PyramidNet-272)
                         # with eta 0.01, a different T, so its radii are not directly comparable with davda54's. Larger
                         # rho = stronger flatness regularization, with a risk of underfitting or instability
SAM_ADAPTIVE = True  # T128 launch copy (was: False)
SAM_ETA = 0.01  # T128 launch copy (was: 0.0)
                         # so weights near zero still get a nonzero radius; 0.0 = davda54's T = |w| (T112)
SAM_M = None             # m-sharpness: None = one perturbation from the whole batch; an integer m dividing BATCH_SIZE
                         # gives each chunk of m images its own perturbation from its own gradient and averages the chunk
                         # gradients for one SGD step. The SAM paper finds smaller m usually generalizes better; ASAM's
                         # PyramidNet used m = 32 of batch 256. Cost: 2 x BATCH_SIZE/m smaller passes (slower); BatchNorm
                         # uses m-image statistics in every pass, and each chunk's first pass updates the running
                         # statistics (second passes never do). Not combinable with MIXSTYLE

# Data and augmentation
SPLIT_SEED = 0           # seed of the train/validation split; keep it fixed so runs are comparable
CUTOUT_SIZE = 16
CROP_PADDING_MODE = "constant"  # T128 launch copy (was: "reflect")
                         # edge pixel, as in scheme C/T31); training random crop only, validation/test unchanged
AUTOAUG_ERASE = "cutout" # occlusion after AutoAugment or TrivialAugment: "cutout" | "erasing" (RandomErasing) | "none"
ERASING_P = 0.5          # RandomErasing probability

# Training-only Mixup option (P13): off in this preset; enabling it also requires AUG="basic" and smoothing 0 above.
MIXUP = False            # True enables P13 batch Mixup (with AUG="basic" and LABEL_SMOOTHING=0.0)
MIXUP_ALPHA = 0.2        # positive Beta distribution parameter; one lambda per triggered batch
MIXUP_PROB = 0.5         # probability of mixing a training batch; validation/test never mix

# Training-only CutMix (Yun et al., 2019), ported from E_pyramid_t106_cutmix.ipynb (T119); off in this preset.
CUTMIX = False           # True: with probability CUTMIX_PROB a batch gets one rectangle from a shuffled partner batch;
                         # targets weighted by the clipped patch area, label smoothing applied to both. Exclusive with
                         # MIXUP. May be combined with Cutout (AUTOAUG_ERASE="cutout" / AUG="cutout"): the CutMix paper
                         # replaces Cutout (T119 turned it off), Fast AA / TA recipes use Cutout and ASAM's PyramidNet
                         # used AutoAugment + CutMix; stacking both is stronger regularization, with a risk of
                         # underfitting on short schedules. SAM passes and SAM_M chunks share the same mixed batch
CUTMIX_ALPHA = 1.0       # Beta(alpha, alpha) area ratio, one draw per triggered batch
CUTMIX_PROB = 0.5        # probability of mixing a training batch; validation/test never mix

# Training-only MixStyle after PyramidNet stages 1 and 2 (section 4); off in this preset. Parameter-free.
MIXSTYLE = False         # True mixes per-image feature mean/std with a shuffled partner in training; identity in evaluation
MIXSTYLE_P = 0.5         # probability that a MixStyle layer mixes a training batch (drawn per layer and step)
MIXSTYLE_ALPHA = 0.1     # Beta(alpha, alpha) mixing weight, one per image
MIXSTYLE_EPS = 1e-6      # added to the feature variance before the square root

# Hardware (None = automatic: on for CUDA, off for MPS / CPU)
DEVICE = "cuda"          # "cuda" | "mps" | "cpu"; None = $CIFAR_DEVICE, else cuda > mps > cpu
CHANNELS_LAST = None
CUDNN_BENCHMARK = None
PIN_MEMORY = None
WORKERS = None           # DataLoader workers for AUG = "autoaug" / "trivialaug" (always 0 on macOS / Windows, see section 2)
EVAL_BATCH_SIZE = 1000
LOG_INTERVAL = 0         # also print every N steps within an epoch (0 = off)
# Optional speed switches (README "Compilation"); neither is part of the recipe or of any fingerprint.
COMPILE = "reduce-overhead"  # T128 launch copy: SAM CUDA-graph path, gated by the launch preflight (fallback "default")
                         # With SAM: "reduce-overhead" = CUDA graphs behind an automatic launch preflight (section 5),
                         # "default" = the reference SAM path; "max-autotune" is rejected; SAM_M / MIXSTYLE need "default"
BENCHMARK_STEPS = 0      # > 0: only time this many training steps (real data, current COMPILE); no run folder; then stop

# Prediction
CHECKPOINT = "last"      # checkpoint used for the submission: "best" (highest validation accuracy) | "last"
WEIGHTS = "ema"          # "auto" chooses on validation using TTA; for VAL_SIZE=0 set "model" or "ema" explicitly
TTA = True               # average the softmax of each test image and its horizontal flip

# Runtime host, independent of the editor: auto | colab | kaggle | local.
# For a VS Code kernel connected to Colab, set "colab" if auto detection is ambiguous.
RUNTIME_ENV = "colab"    # explicit value overrides CIFAR_ENV; auto uses CIFAR_ENV then detection

# Paths (None = $CIFAR_DATA_DIR / $CIFAR_CACHE_DIR / $CIFAR_RESULT_DIR, else the default of the environment)
AUTO_DOWNLOAD = False     # Kaggle download/login code is commented out; use uploaded data
SAVE_CACHE_TO_DRIVE = True # on Colab, automatically save/verify MyDrive/DL_Project1/cache.zip
DATA_DIR = None          # Colab: MyDrive/DL_Project1/data/ (folder or override with a ZIP path)
CACHE_DIR = None
RESULT_DIR = None
SHOW_SUMMARY = True      # torchinfo layer table in section 4 (skipped if torchinfo is not installed)

# Architecture switches (requirement 15): DROP_PATH, SHAKEDROP, BLOCK, DEPTH and ALPHA (section 4).
TASK_ID = "T128"  # T128 launch copy (was: "T120")
MODEL_VERSION = None     # allocate the next E version at actual launch; no advance reservations
E_VERSION_FLOOR = 32  # T128 launch copy (was: 19)
                         # hold newer E runs. Launch T120 and T121 one after the other (each takes the next version)
DROP_PATH = 0.0          # stochastic depth: block l of L dropped with prob DROP_PATH*l/L; must be 0 with ShakeDrop
SHAKEDROP = 0.5          # ShakeDrop: block l of L perturbed with prob SHAKEDROP*l/L (p_L = 1 - SHAKEDROP); 0 = off
BLOCK = "bottleneck"  # T128 launch copy (was: "basic")
DEPTH = 110              # basic: 6n+2 (110 = 18 blocks per stage); bottleneck: 9n+2 (164, 200, 272)
ALPHA = 128  # T128 launch copy (was: 98)
# Inference overrides only; saved checkpoint scientific arguments remain authoritative.
SAVE_CACHE_TO_DRIVE = False
SHOW_SUMMARY = False
EVAL_BATCH_SIZE = 256
COMPILE = "off"  # short fp32 inference, no training compilation


In [19]:
import copy
import csv
import hashlib
import importlib.util
import importlib.metadata
import io
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import tempfile
import zlib
import zipfile
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from matplotlib.ticker import MaxNLocator
from PIL import Image

ENVS = ("local", "colab", "kaggle")
KAGGLE_SLUG = "cifar-10-res-net-architecture-challenge-dl"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/DL_Project1"
WORK_DIRS = {"colab": "/content", "kaggle": "/kaggle/working"}
COLAB_CKPT_EVERY = 5  # Drive writes are slow; best-ckpt.pt and the final epoch are always saved
PROJECT_DIR = Path.cwd().resolve()
LOCAL_PROJECT_DIR = PROJECT_DIR.parent if PROJECT_DIR.name == "Archive" else PROJECT_DIR
# Start method for the AutoAugment DataLoader workers: "fork" lets them use the Dataset class defined in this
# notebook. Under "spawn" (macOS, Windows) they cannot import it, so the pipeline runs in the main process there.
MP_CONTEXT = "fork" if sys.platform.startswith("linux") else None

_warned = set()


def _getenv(name):
    value = os.environ.get(name, "").strip()
    return value or None


def warn_once(msg):
    if msg not in _warned:
        _warned.add(msg)
        print(f"Warning: {msg}", flush=True)


def _module_exists(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


def detect_env():
    """Detect the kernel host, independent of the notebook editor (browser or VS Code)."""
    configured = str(globals().get("RUNTIME_ENV") or "auto").strip().lower()
    if configured not in ("auto",) + ENVS:
        raise ValueError(f"RUNTIME_ENV={configured!r}; expected auto or one of {ENVS}")
    requested = configured if configured != "auto" else (_getenv("CIFAR_ENV") or "auto").lower()
    if requested not in ("auto",) + ENVS:
        raise ValueError(f"CIFAR_ENV={requested!r}; expected auto or one of {ENVS}")
    if requested != "auto":
        return requested
    # Runtime markers beat directory names: /kaggle/input may also exist on Colab.
    if any(_getenv(key) for key in ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION")):
        return "colab"
    if _getenv("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if _module_exists("google.colab"):
        return "colab"
    # A directory alone is weak evidence; use the full layout only as a fallback.
    if os.path.isdir("/kaggle/input") and os.path.isdir("/kaggle/working"):
        return "kaggle"
    return "local"


def work_dir(env=None):
    env = env or detect_env()
    return Path(_getenv("CIFAR_WORK_DIR") or WORK_DIRS.get(env, PROJECT_DIR))


def drive_dir():
    return Path(_getenv("CIFAR_DRIVE_DIR") or DRIVE_PROJECT_DIR)


def drive_mounted():
    return drive_dir().parent.is_dir()  # .../MyDrive exists once Drive is mounted


def kaggle_input_dir():
    return Path(_getenv("CIFAR_KAGGLE_INPUT") or "/kaggle/input")


def find_competition_data(root=None, max_depth=6):
    """Directories under `root` that contain train_labels.csv, best match first.

    Breadth-first, follows symlinks, skips *_images folders (50k files) and hidden folders.
    """
    root = Path(root or kaggle_input_dir())
    found, frontier = [], [(root, 0)]
    while frontier:
        d, depth = frontier.pop(0)
        try:
            entries = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError:
            continue
        if any(e.name == "train_labels.csv" and e.is_file() for e in entries):
            found.append(d)
        if depth < max_depth:
            frontier += [(Path(e.path), depth + 1) for e in entries
                         if e.is_dir() and not e.name.endswith("_images") and not e.name.startswith(".")]

    def rank(d):
        complete = (d / "test.csv").is_file() and (d / "sample_submission.csv").is_file()
        return (not complete, KAGGLE_SLUG not in str(d), len(d.parts), str(d))

    return sorted(found, key=rank)


@dataclass
class Paths:
    env: str
    data_dir: Path
    cache_dir: Path
    result_dir: Path


def resolve_paths(data_dir=None, cache_dir=None, result_dir=None, env=None):
    """Data, cache and runs directories: argument > environment variable > environment default."""
    env = env or detect_env()
    data = data_dir or _getenv("CIFAR_DATA_DIR")
    cache = cache_dir or _getenv("CIFAR_CACHE_DIR")
    runs = result_dir or _getenv("CIFAR_RESULT_DIR")
    if env == "local":
        data = data or LOCAL_PROJECT_DIR / "data"
        cache = cache or LOCAL_PROJECT_DIR / "cache"
        runs = runs or LOCAL_PROJECT_DIR / "result"
    elif env == "colab":
        cache = cache or work_dir(env) / "cache"
        data = data or drive_dir() / "data"  # manually uploaded ZIPs or extracted competition files
        if runs is None:
            if drive_mounted():
                runs = drive_dir() / "result"
            else:
                runs = work_dir(env) / "result"
                warn_once(f"Google Drive is not mounted ({drive_dir().parent} missing); writing results to "
                          f"{runs}, which is lost when the runtime is recycled")
    else:  # kaggle
        cache = cache or work_dir(env) / "cache"
        runs = runs or work_dir(env) / "result"
        if data is None:
            found = find_competition_data()
            if found:
                data = found[0]
            else:
                data = work_dir(env) / "data"
                warn_once(f"no train_labels.csv found under {kaggle_input_dir()}; checking cache or manually supplied data")
    return Paths(env, Path(data), Path(cache), Path(runs))


def default_ckpt_every(env=None):
    return COLAB_CKPT_EVERY if (env or detect_env()) == "colab" else 1


def available_cpus():
    try:
        return len(os.sched_getaffinity(0))
    except AttributeError:  # macOS
        return os.cpu_count() or 1


def default_workers(device_type):
    """DataLoader workers (used by the per-image AutoAugment pipeline only)."""
    n = available_cpus()
    if device_type == "cuda":
        return min(8, max(2, n - 1))
    return min(4, max(1, n // 2))  # 4 on the 12-core M4 Pro, as before


def get_device(name=None):
    """`name` (DEVICE), else $CIFAR_DEVICE, else cuda > mps > cpu."""
    name = name or _getenv("CIFAR_DEVICE") or "auto"
    if name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(name)


def cuda_bf16_native(device):
    return torch.cuda.get_device_capability(device) >= (8, 0)  # Ampere or newer (L4, A100, ...)


def resolve_amp(requested, device):
    """'bf16', 'fp16' or 'off'. requested: auto | on | off | bf16 | fp16 (bools accepted)."""
    req = {True: "on", False: "off", None: "auto"}.get(requested, requested)
    req = str(req).lower()
    if req == "off":
        return "off"
    if device.type != "cuda":
        if req != "auto":
            # MPS: fp16 autocast gave no speed-up on an M4 Pro, and torch 2.5.1 GradScaler fails on MPS.
            print(f"Warning: AMP is only used on CUDA; training in fp32 on {device.type}.")
        return "off"
    if req == "fp16":
        return "fp16"
    if cuda_bf16_native(device):
        return "bf16"
    if req == "bf16":
        print("Warning: this GPU has no native bf16; using fp16 autocast + GradScaler instead.")
    return "fp16"


# torch.compile modes for the training step (README "Compilation"); a speed option, never part of the recipe.
COMPILE_MODES = ("off", "default", "reduce-overhead", "max-autotune", "max-autotune-no-cudagraphs")


def resolve_compile(requested, device, name="COMPILE"):
    """One of COMPILE_MODES. Compilation is used on CUDA only; elsewhere it warns and runs uncompiled ("off")."""
    if not isinstance(requested, str) or requested not in COMPILE_MODES:
        raise ValueError(f"{name} = {requested!r}; choose one of {COMPILE_MODES}")
    if requested != "off" and device.type != "cuda":
        print(f"Warning: {name}={requested!r} is only used on CUDA; running uncompiled on {device.type}.")
        return "off"
    return requested


def resolve_hardware(args, device):
    """Fill in the automatic (None) speed settings on `args` for `device`, in place."""
    cuda = device.type == "cuda"
    args.amp = resolve_amp(getattr(args, "amp", "auto"), device)
    args.compile = resolve_compile(getattr(args, "compile", "off"), device)
    if args.channels_last is None:
        args.channels_last = cuda
    elif args.channels_last and device.type == "mps":
        # torch 2.5.1 MPS: backward through conv/BN/ReLU in channels_last raises a stride error.
        print("Warning: CHANNELS_LAST is not supported on MPS with this PyTorch; disabling it.")
        args.channels_last = False
    if getattr(args, "cudnn_benchmark", None) is None:
        args.cudnn_benchmark = cuda
    if getattr(args, "pin_memory", None) is None:
        args.pin_memory = cuda
    if getattr(args, "workers", None) is None:
        args.workers = default_workers(device.type)
    if MP_CONTEXT is None:
        args.workers = 0  # see MP_CONTEXT
    return args


def device_name(device):
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(device)
        return f"cuda ({props.name}, {props.total_memory / 2**30:.0f} GB)"
    if device.type == "cpu":
        return f"cpu ({available_cpus()} cores)"
    return f"{device.type} ({platform.machine()})"


def _on(x):
    return "on" if x else "off"


def summary_line(paths, device=None, args=None):
    """One line: environment, device, torch, speed settings and the output directories."""
    import torch

    parts = [f"env {paths.env}"]
    if device is not None:
        parts.append(device_name(device))
    parts.append(f"torch {torch.__version__}")
    if args is not None:
        parts.append(f"amp {args.amp} | channels_last {_on(args.channels_last)} | "
                     f"cudnn.benchmark {_on(args.cudnn_benchmark)} | workers {args.workers} | "
                     f"pin_memory {_on(args.pin_memory)}")
        if getattr(args, "compile", "off") != "off":
            parts.append(f"compile {args.compile}")
    parts.append(f"data {paths.data_dir} | cache {paths.cache_dir} | result {paths.result_dir}")
    return "[env] " + " | ".join(parts)

def run_timestamp():
    return datetime.now(ZoneInfo("America/New_York")).strftime("%m-%d-%H-%M")


# Exact historical aliases: checkpoint bytes and signed export manifests remain unchanged.
RUN_ID_ALIASES = {'10-30-08-40_A_resnet-ver-1-cutout-ema': '10-30-08-40_A_resnet-ver-1-T00-cutout-ema', '10-30-09-44_B_wrn-ver-1-cutout-ema': '10-30-09-44_B_wrn-ver-1-T00-cutout-ema', '10-30-09-46_C_seresnet-ver-1-cutout-ema': '10-30-09-46_C_seresnet-ver-1-T00-cutout-ema', '10-30-09-50_C_seresnet-ver-2-cutout-ema': '10-30-09-50_C_seresnet-ver-2-T00-cutout-ema', '10-30-19-41_B_wrn-ver-2-wide-ema': '10-30-19-41_B_wrn-ver-2-T08-wide-ema', '10-30-20-03_A_resnet-ver-6-eca-ema': '10-30-20-03_A_resnet-ver-2-T48-eca-ema', '10-30-20-04_C_seresnet-ver-8-se8-ema': '10-30-20-04_C_seresnet-ver-3-T06-se8-ema', '10-30-21-05_E_pyramid-ver-1-basic110-ema': '10-30-21-05_E_pyramid-ver-1-T18-basic110-ema', '10-30-21-07_G_resnext-ver-1-grouped-ema': '10-30-21-07_G_resnext-ver-1-T42-grouped-ema', '10-30-21-17_A_resnet-ver-7-mixup-ema': '10-30-21-17_A_resnet-ver-3-T49-mixup-ema', '10-30-21-18_B_wrn-ver-8-mixup-ema': '10-30-21-18_B_wrn-ver-3-T50-mixup-ema', '10-30-21-42_F_shake-ver-1-shake40': '10-30-21-42_F_shake-ver-1-T35-shake40', '10-30-21-47_C_seresnet-ver-31-mixup-ema': '10-30-21-47_C_seresnet-ver-4-T51-mixup-ema', '10-30-23-09_A_resnet-ver-4-early-ema': '10-30-23-09_A_resnet-ver-4-T22-early-ema', '10-30-23-10_B_wrn-ver-6-deep-ema': '10-30-23-10_B_wrn-ver-4-T24-deep-ema', '10-30-23-43_C_seresnet-ver-3-autoaug-cutout': '10-30-23-43_C_seresnet-ver-5-T01-autoaug-cutout', '10-30-23-56_A_resnet-ver-5-mid-ema': '10-30-23-56_A_resnet-ver-5-T26-mid-ema', '10-31-00-42_B_wrn-ver-5-balanced-ema': '10-31-00-42_B_wrn-ver-5-T23-balanced-ema', '10-31-00-50_C_seresnet-ver-19-reflect-ema': '10-31-00-50_C_seresnet-ver-6-T31-reflect-ema', '10-30-22-34_F_shake-ver-2-shake40-1800ep': '10-30-22-34_F_shake-ver-2-T35-shake40-1800ep', '10-31-20-06_E_pyramid-ver-3-T61-autoaug-only': '10-31-20-06_E_pyramid-ver-3-T69-autoaug-only'}


def canonical_run_id(run_id):
    return RUN_ID_ALIASES.get(run_id, run_id)


def run_task_id(run_id):
    match = re.search(r"-ver-\d+-(T\d{2,})-", canonical_run_id(run_id))
    return match.group(1) if match else "T00"


def run_paths(result_root, run_id, create=False):
    """One timestamped ID across directories grouped by artifact type."""
    if not run_id or Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("Run ID must be a single directory name")
    run_id = canonical_run_id(run_id)
    root = Path(result_root)
    folders = {"checkpoints": "checkpoints", "configs": "configs", "logs": "logs",
               "validation": "evaluations/validation", "test": "evaluations/test",
               "plots": "plots", "progress": "progress", "manifests": "manifests", "exports": "exports"}
    paths = SimpleNamespace(root=root, run_id=run_id,
                            **{key: root / folder / run_id for key, folder in folders.items()})
    if not paths.checkpoints.exists():
        for legacy, canonical in RUN_ID_ALIASES.items():
            old_dir = root / "checkpoints" / legacy
            if canonical == run_id and old_dir.is_dir():
                paths.checkpoints = old_dir
                break
    if create:
        for key in folders:
            getattr(paths, key).mkdir(parents=True, exist_ok=True)
    return paths


def paths_for_run(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    if checkpoint_dir.parent.name != "checkpoints":
        raise ValueError("Expected result/checkpoints/<RUN_ID>; legacy flat runs must be migrated explicitly")
    return run_paths(checkpoint_dir.parent.parent, checkpoint_dir.name)


def artifact_path(artifacts, category, role, extension, event=False, unique=False):
    """Stable training paths; dated evaluation/export paths with collision-safe role suffixes."""
    folder = getattr(artifacts, category)
    prefix = artifacts.run_id
    if event:
        identifier = artifacts.run_id.split('_', 1)[1] if '_' in artifacts.run_id else artifacts.run_id
        prefix = f"{run_timestamp()}_{identifier}"
    stem = f"{prefix}_{role}"
    path = folder / (stem + extension)
    if unique:
        index = 1
        while any(folder.glob(path.stem + '*')):
            index += 1
            path = folder / (f"{stem}-{index}" + extension)
    return path


def checkpoint_path(run_dir, which):
    if which not in ('best', 'last'):
        raise ValueError("Checkpoint role must be best or last")
    run_dir = Path(run_dir)
    canonical = canonical_run_id(run_dir.name)
    candidates = [run_dir.parent / canonical / f"{canonical}_{which}-ckpt.pt",
                  run_dir / f"{run_dir.name}_{which}-ckpt.pt"]
    for legacy, target in RUN_ID_ALIASES.items():
        if target == canonical:
            candidates.extend([run_dir.parent / canonical / f"{legacy}_{which}-ckpt.pt",
                               run_dir.parent / legacy / f"{legacy}_{which}-ckpt.pt"])
    return next((p for p in candidates if p.is_file()), candidates[0])


In [20]:
ENV = detect_env()
print(f"[runtime] detected {ENV} | kernel {sys.executable} | cwd {Path.cwd()} | "
      f"RUNTIME_ENV={RUNTIME_ENV!r} | CIFAR_ENV={_getenv('CIFAR_ENV')!r}")
if ENV == "colab":
    try:
        from google.colab import drive
    except ImportError:  # CIFAR_ENV=colab outside Google Colab (testing)
        print("google.colab is not available: skipping drive.mount")
    else:
        drive.mount("/content/drive")

PATHS = resolve_paths(DATA_DIR, CACHE_DIR, RESULT_DIR, ENV)
DEV = get_device(DEVICE)
if ENV == "colab" and DEV.type != "cuda" and (DEVICE or _getenv("CIFAR_DEVICE")) != "cpu":
    raise RuntimeError("No GPU: choose Runtime > Change runtime type > A100 GPU, then Runtime > Run all again")
HW = resolve_hardware(SimpleNamespace(amp=AMP, channels_last=CHANNELS_LAST, cudnn_benchmark=CUDNN_BENCHMARK,
                                      pin_memory=PIN_MEMORY, workers=WORKERS), DEV)
print(summary_line(PATHS, DEV, HW))
print(f"python {platform.python_version()} | torchvision {torchvision.__version__} | numpy {np.__version__} | "
      f"{platform.platform()} | {available_cpus()} CPUs")
if ENV == "colab":
    print(f"Google Drive folder {drive_dir()}: {'found' if drive_dir().is_dir() else 'NOT FOUND'}")
if ENV == "kaggle":
    print(f"Kaggle input {kaggle_input_dir()}: train_labels.csv in "
          f"{[str(d) for d in find_competition_data()] or 'none'}")
if DEV.type == "cuda" and shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert DEV.type == "cuda"
assert "A100" in torch.cuda.get_device_name(0)
assert 38 < torch.cuda.get_device_properties(0).total_memory / 2**30 < 42


[runtime] detected colab | kernel /usr/bin/python3 | cwd /content | RUNTIME_ENV='colab' | CIFAR_ENV=None
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[env] env colab | cuda (NVIDIA A100-SXM4-40GB, 39 GB) | torch 2.11.0+cu130 | amp bf16 | channels_last on | cudnn.benchmark on | workers 8 | pin_memory on | data /content/drive/MyDrive/DL_Project1/data | cache /content/cache | result /content/drive/MyDrive/DL_Project1/result
python 3.13.16 | torchvision 0.26.0+cu130 | numpy 2.4.6 | Linux-6.6.159+-x86_64-with-glibc2.39 | 12 CPUs
Google Drive folder /content/drive/MyDrive/DL_Project1: found
Wed Nov  4 19:29:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name     

In [21]:
NUM_CLASSES = 10
CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
CACHE_FORMAT = 2
SOURCE_CSVS = ("train_labels.csv", "test.csv", "sample_submission.csv")
CACHE_ARRAYS = ("train_images.npy", "train_labels.npy", "train_ids.npy", "test_images.npy", "test_ids.npy")


def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _decode_pngs(paths):
    def load(p):
        with Image.open(p) as im:
            return np.asarray(im.convert("RGB"), dtype=np.uint8)

    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as pool:
        return np.stack(list(pool.map(load, paths, chunksize=256)))


def _save_npy(path, arr):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)


def _copy_csvs(data_dir, cache_dir):
    if data_dir.resolve() == cache_dir.resolve():
        return
    for name in SOURCE_CSVS:
        tmp = cache_dir / (name + ".tmp")
        shutil.copyfile(data_dir / name, tmp)
        os.replace(tmp, cache_dir / name)


def _write_meta(cache_dir, n_train, n_test, shape, note):
    files = {name: {"sha256": sha256_file(cache_dir / name), "bytes": (cache_dir / name).stat().st_size}
             for name in CACHE_ARRAYS + SOURCE_CSVS}
    fingerprint = hashlib.sha256("".join(f"{n} {files[n]['sha256']}\n" for n in sorted(files)).encode())
    meta = {"format": CACHE_FORMAT, "n_train": n_train, "n_test": n_test, "shape": list(shape),
            "fingerprint": fingerprint.hexdigest(), "files": files, "note": note,
            "written": datetime.now().isoformat(timespec="seconds")}
    tmp = cache_dir / "meta.json.tmp"
    tmp.write_text(json.dumps(meta, indent=2))
    os.replace(tmp, cache_dir / "meta.json")  # written last: a cache without it is incomplete
    return meta


def read_meta(cache_dir):
    path = Path(cache_dir) / "meta.json"
    return json.loads(path.read_text()) if path.exists() else None


def build_cache(data_dir, cache_dir):
    """Decode all PNGs once into uint8 arrays (N, 32, 32, 3) and save them with the CSVs and meta.json."""
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    train = pd.read_csv(data_dir / "train_labels.csv")
    print(f"[cache] decoding {len(train):,} training PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "train_images" / f for f in train["filename"]])
    _save_npy(cache_dir / "train_images.npy", x)
    _save_npy(cache_dir / "train_labels.npy", train["label"].to_numpy(np.int64))
    _save_npy(cache_dir / "train_ids.npy", train["id"].to_numpy(np.int64))

    test = pd.read_csv(data_dir / "test.csv")
    print(f"[cache] decoding {len(test):,} test PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "test_images" / f for f in test["filename"]])
    _save_npy(cache_dir / "test_images.npy", x)
    _save_npy(cache_dir / "test_ids.npy", test["id"].to_numpy(np.int64))

    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), x.shape[1:], f"built from {data_dir}")
    print(f"[cache] wrote {cache_dir} (fingerprint {meta['fingerprint'][:12]})", flush=True)
    return meta


def _upgrade_format1(data_dir, cache_dir):
    """Add CSV copies and hashes to a format-1 cache if its arrays match the CSVs. Returns success."""
    train = pd.read_csv(data_dir / "train_labels.csv")
    test = pd.read_csv(data_dir / "test.csv")
    try:
        x_train = np.load(cache_dir / "train_images.npy", mmap_mode="r")
        x_test = np.load(cache_dir / "test_images.npy", mmap_mode="r")
        ok = (x_train.shape[0] == len(train) and x_test.shape[0] == len(test)
              and x_train.shape[1:] == x_test.shape[1:]
              and np.array_equal(np.load(cache_dir / "train_labels.npy"), train["label"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "train_ids.npy"), train["id"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "test_ids.npy"), test["id"].to_numpy(np.int64)))
    except (OSError, ValueError):
        ok = False
    if not ok:
        return False
    shape = x_train.shape[1:]
    del x_train, x_test
    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), shape, f"format 1 cache upgraded with {data_dir}")
    print(f"[cache] upgraded {cache_dir} to format {CACHE_FORMAT} (arrays unchanged, "
          f"fingerprint {meta['fingerprint'][:12]})", flush=True)
    return True


def _cache_problem(meta, cache_dir, data_dir, has_csvs):
    """None if the format-2 cache is complete and matches the CSVs in data_dir, else the reason."""
    for name, info in meta["files"].items():
        path = cache_dir / name
        if not path.exists() or path.stat().st_size != info["bytes"]:
            return f"{name} is missing or has the wrong size"
    if has_csvs and data_dir.resolve() != cache_dir.resolve():
        for name in SOURCE_CSVS:
            if sha256_file(data_dir / name) != meta["files"][name]["sha256"]:
                return f"{data_dir / name} differs from the cached copy"
    return None


def ensure_cache(data_dir, cache_dir):
    """Return the cache dir, building, upgrading or rebuilding the cache only when needed.

    A complete format-2 cache is used as is; if data_dir also holds the CSVs, they must match the
    cached copies (otherwise the cache is rebuilt from the PNGs).
    """
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    has_csvs = all((data_dir / n).is_file() for n in SOURCE_CSVS)
    has_pngs = has_csvs and (data_dir / "train_images").is_dir() and (data_dir / "test_images").is_dir()
    meta = read_meta(cache_dir)
    if meta is not None and meta.get("format") == CACHE_FORMAT:
        problem = _cache_problem(meta, cache_dir, data_dir, has_csvs)
        if problem is None:
            return cache_dir
        print(f"[cache] {problem}; rebuilding", flush=True)
    elif meta is not None and has_csvs and all((cache_dir / n).exists() for n in CACHE_ARRAYS):
        if _upgrade_format1(data_dir, cache_dir):
            return cache_dir
        print("[cache] format 1 cache does not match the CSVs; rebuilding", flush=True)
    if not has_pngs and meta is not None and meta.get("format") is None:
        raise FileNotFoundError(
            f"{cache_dir} is a format-1 cache and {data_dir} has no CSVs to upgrade it with; "
            f"set DATA_DIR to the folder with the competition CSVs")
    if not has_pngs:
        raise FileNotFoundError(
            f"no usable cache in {cache_dir} and no PNG images in {data_dir} to build one; set DATA_DIR "
            f"(folder with train_labels.csv, train_images/, ...) or CACHE_DIR, or on Colab upload cache.zip "
            f"to {drive_dir()}")
    build_cache(data_dir, cache_dir)
    return cache_dir


def verify_cache(cache_dir):
    """Check every cached file against the sha256 in meta.json; return meta (raises on a mismatch)."""
    cache_dir = Path(cache_dir)
    meta = read_meta(cache_dir)
    _check_cache_manifest(meta)
    bad = [n for n, info in meta["files"].items()
           if not (cache_dir / n).exists() or sha256_file(cache_dir / n) != info["sha256"]]
    if bad:
        raise RuntimeError(f"cache files do not match meta.json: {bad}")
    n_train, n_test = validate_cache_arrays(cache_dir)
    if (meta["n_train"], meta["n_test"], meta["shape"]) != (n_train, n_test, [32, 32, 3]):
        raise RuntimeError("Cache metadata does not match its arrays")
    return meta


def validate_cache_arrays(cache_dir):
    """Validate shapes, labels, IDs, and the correspondence between arrays and CSVs."""
    cache_dir = Path(cache_dir)
    frames = {name: pd.read_csv(cache_dir / name) for name in SOURCE_CSVS}
    for name, columns in (("train_labels.csv", ("id", "filename", "label")),
                          ("test.csv", ("id", "filename")),
                          ("sample_submission.csv", ("id", "label"))):
        frame = frames[name]
        if not set(columns).issubset(frame.columns) or frame.empty:
            raise ValueError(f"{name}: missing columns or no rows")
        if not pd.api.types.is_integer_dtype(frame["id"]) or not frame["id"].is_unique:
            raise ValueError(f"{name}: IDs must be unique integers")
        if "filename" in frame and (frame["filename"].isna().any() or not frame["filename"].is_unique):
            raise ValueError(f"{name}: filenames must be present and unique")
    train, test = frames["train_labels.csv"], frames["test.csv"]
    if not pd.api.types.is_integer_dtype(train["label"]) or not train["label"].between(0, 9).all():
        raise ValueError("Training labels must be integers in 0..9")
    for split, frame in (("train", train), ("test", test)):
        images = np.load(cache_dir / f"{split}_images.npy", mmap_mode="r", allow_pickle=False)
        ids = np.load(cache_dir / f"{split}_ids.npy", allow_pickle=False)
        if images.dtype != np.uint8 or images.shape != (len(frame), 32, 32, 3):
            raise ValueError(f"{split} images must be uint8 NHWC with shape (N,32,32,3)")
        if not np.array_equal(ids, frame["id"].to_numpy()):
            raise ValueError(f"{split} array IDs differ from CSV order")
    labels = np.load(cache_dir / "train_labels.npy", allow_pickle=False)
    if not np.issubdtype(labels.dtype, np.integer) or not np.array_equal(labels, train["label"].to_numpy()):
        raise ValueError("Cached labels differ from training CSV")
    if set(frames["sample_submission.csv"]["id"]) != set(test["id"]):
        raise ValueError("Sample submission and test IDs differ")
    return len(train), len(test)


def cache_fingerprint(cache_dir):
    meta = read_meta(cache_dir) or {}
    return meta.get("fingerprint")


def load_train(data_dir, cache_dir, mmap=False):
    cache_dir = ensure_cache(data_dir, cache_dir)
    images = np.load(cache_dir / "train_images.npy", mmap_mode="r" if mmap else None)
    return images, np.load(cache_dir / "train_labels.npy"), np.load(cache_dir / "train_ids.npy")


def load_test(data_dir, cache_dir):
    cache_dir = ensure_cache(data_dir, cache_dir)
    return np.load(cache_dir / "test_images.npy"), np.load(cache_dir / "test_ids.npy")


def _cache_matches_source(meta, source_dir):
    """An explicitly supplied set of CSVs must agree with a reused cache."""
    source_dir = Path(source_dir)
    return all(not (source_dir / name).is_file() or
               sha256_file(source_dir / name) == meta['files'][name]['sha256'] for name in SOURCE_CSVS)


def _check_cache_manifest(meta):
    required = set(CACHE_ARRAYS + SOURCE_CSVS)
    if (not isinstance(meta, dict) or meta.get('format') != CACHE_FORMAT
            or not isinstance(meta.get('files'), dict) or set(meta['files']) != required):
        raise ValueError('Cache needs a format-2 manifest with all required files')
    for info in meta['files'].values():
        if (not isinstance(info, dict) or not isinstance(info.get('bytes'), int)
                or info['bytes'] <= 0 or not isinstance(info.get('sha256'), str)
                or not re.fullmatch(r'[0-9a-f]{64}', info['sha256'])):
            raise ValueError('Invalid cache file size or SHA256 in manifest')
    fingerprint = hashlib.sha256(''.join(
        f"{name} {meta['files'][name]['sha256']}\n" for name in sorted(required)).encode()).hexdigest()
    if fingerprint != meta.get('fingerprint'):
        raise ValueError('Cache ZIP fingerprint differs from its manifest')
    return required


def verify_cache_zip(path):
    """Check ZIP contents and every uncompressed file hash, not just the ZIP header."""
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        if len(names) != len(set(names)) or set(names) != set(CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',)):
            raise ValueError('Cache ZIP must contain exactly the required files at its root')
        meta = json.loads(z.read('meta.json'))
        for name in _check_cache_manifest(meta):
            info = meta['files'][name]
            digest = hashlib.sha256()
            with z.open(name) as f:
                for block in iter(lambda: f.read(1 << 22), b''):
                    digest.update(block)
            if z.getinfo(name).file_size != info['bytes'] or digest.hexdigest() != info['sha256']:
                raise ValueError(f'Cache ZIP checksum mismatch: {name}')
    return meta


def extract_zip_checked(archive, destination):
    """Extract inside a staging folder, rejecting escaping paths and symlink entries."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if ('\\' in member.filename or not target.is_relative_to(root)
                    or (member.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        z.extractall(destination)  # decompression also checks each member's CRC


def publish_data_directory(staged, destination):
    """Keep the previous directory until the validated replacement can be published."""
    staged, destination = Path(staged), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    previous = destination.with_name(f'.{destination.name}.previous-{time.time_ns()}')
    if destination.exists():
        os.replace(destination, previous)
    try:
        os.replace(staged, destination)
    except BaseException:
        if previous.exists():
            try:
                os.replace(previous, destination)
            except OSError:
                # Keep recovery files even if the filesystem also refuses rollback.
                print(f'[data] previous files preserved at {previous}; restore them before retrying')
        raise
    else:
        if previous.is_dir() and not previous.is_symlink():
            shutil.rmtree(previous)
        else:
            previous.unlink(missing_ok=True)


def drive_cache_candidates(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return []
    pattern = re.compile(r'cache(?: \(\d+\))?\.zip$')
    return sorted((p for p in folder.iterdir() if p.is_file() and pattern.fullmatch(p.name)),
                  key=lambda p: (p.stat().st_mtime_ns, p.name), reverse=True)


def fetch_drive_cache(folder, cache_dir, source_dir=None):
    """Try Drive cache copies newest first; never replace a cache with an unverified ZIP."""
    cache_dir = Path(cache_dir)
    cache_dir.parent.mkdir(parents=True, exist_ok=True)
    for archive in drive_cache_candidates(folder):
        try:
            with tempfile.TemporaryDirectory(prefix='.restore-cache-', dir=cache_dir.parent) as tmp:
                local_zip, staged = Path(tmp) / 'cache.zip', Path(tmp) / 'cache'
                shutil.copyfile(archive, local_zip)
                meta = verify_cache_zip(local_zip)
                if source_dir is not None and not _cache_matches_source(meta, source_dir):
                    print(f'[cache] skipping {archive.name}: source CSVs differ')
                    continue
                extract_zip_checked(local_zip, staged)
                verify_cache(staged)
                publish_data_directory(staged, cache_dir)
            print(f'[cache] restored {archive} -> {cache_dir}')
            return cache_dir
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError) as exc:
            print(f'[cache] cannot reuse {archive.name}: {type(exc).__name__}; trying another source')
    raise FileNotFoundError('No valid compatible Drive cache was found')


def save_drive_cache(cache_dir, folder):
    """Publish a verified cache.zip once; skip writing an identical valid backup."""
    cache_dir, folder = Path(cache_dir), Path(folder)
    meta = verify_cache(cache_dir)
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / 'cache.zip'
    if destination.is_file():
        try:
            if verify_cache_zip(destination)['fingerprint'] == meta['fingerprint']:
                print(f'[cache] Drive backup already verified: {destination}')
                return destination
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError):
            print('[cache] replacing an invalid Drive backup after verifying its replacement')
    partial = folder / f'.cache-{time.time_ns()}.zip.tmp'
    try:
        with tempfile.TemporaryDirectory(prefix='.export-cache-', dir=cache_dir.parent) as tmp:
            archive = Path(tmp) / 'cache.zip'
            with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as z:
                for name in CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',):
                    z.write(cache_dir / name, arcname=name)
            verify_cache_zip(archive)
            shutil.copyfile(archive, partial)
            verify_cache_zip(partial)  # check the Drive copy before replacing the old backup
            os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)
    print(f'[cache] saved reusable Drive backup: {destination}')
    return destination


def find_raw_data(root, require_images=False):
    """Find CSVs under a wrapper folder, optionally checking every referenced PNG exists."""
    for folder in find_competition_data(root):
        if not all((folder / name).is_file() for name in SOURCE_CSVS):
            continue
        if not require_images:
            return folder
        complete = True
        for split, csv_name in (('train', 'train_labels.csv'), ('test', 'test.csv')):
            frame = pd.read_csv(folder / csv_name)
            if 'filename' not in frame or frame.empty or frame['filename'].isna().any():
                complete = False
                break
            image_root = (folder / f'{split}_images').resolve()
            for filename in frame['filename']:
                image = (image_root / str(filename)).resolve()
                if not image.is_relative_to(image_root) or not image.is_file():
                    complete = False
                    break
            if not complete:
                break
        if complete:
            return folder
    return None


# Kaggle download and login disabled: use manually uploaded data instead.
# def run_kaggle_download():
#     """Use KaggleHub in this notebook process and honor its returned cache/input path."""
#     if not _getenv('KAGGLE_CONFIG_DIR') and (Path('/content') / 'kaggle.json').is_file():
#         os.environ['KAGGLE_CONFIG_DIR'] = '/content'
#     try:
#         kagglehub = importlib.import_module('kagglehub')
#     except ModuleNotFoundError as exc:
#         if exc.name != 'kagglehub':
#             raise RuntimeError('A KaggleHub dependency is missing. Run %pip install -U "kagglehub>=1,<2", '
#                                'restart the kernel, and rerun from the configuration cell.') from None
#         print('[download] installing KaggleHub in this kernel environment')
#         installed = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
#                                     '--disable-pip-version-check', 'kagglehub>=1,<2'],
#                                    capture_output=True, text=True)
#         if installed.returncode:
#             raise RuntimeError('Cannot install KaggleHub. Check internet access and use Python 3.11+; '
#                                'alternatively provide extracted data or a saved cache.')
#         importlib.invalidate_caches()
#         kagglehub = importlib.import_module('kagglehub')
#     print(f'[download] KaggleHub competition_download({KAGGLE_SLUG!r})', flush=True)
#     try:
#         # Do not assume a ZIP filename or pass newer optional API arguments.
#         # This also preserves credentials set by kagglehub.login() in this kernel.
#         downloaded = Path(kagglehub.competition_download(KAGGLE_SLUG))
#     except Exception as exc:
#         response = getattr(exc, 'response', None)
#         status = getattr(response, 'status_code', None)
#         if not isinstance(status, int):
#             status = getattr(exc, 'error_code', None)
#         status_text = f', HTTP {status}' if isinstance(status, int) and 100 <= status <= 599 else ''
#         kind = type(exc).__name__
#         if status == 401 or kind in ('UnauthenticatedError', 'CredentialError'):
#             print('[auth] Kaggle login is needed for this download. Enter your own API token in the login prompt.')
#             try:
#                 kagglehub.login()
#             except Exception as login_error:
#                 raise RuntimeError(
#                     f'Cannot open/complete Kaggle login ({type(login_error).__name__}). '
#                     'For notebook widgets, install ipywidgets and enable widget support in your editor, '
#                     'or use a masked getpass prompt to set KAGGLE_API_TOKEN (see README.md). '
#                     'Then rerun this data cell.') from None
#             # Notebook login displays an asynchronous widget. The kernel must become idle
#             # for its Login button to work; do not block/poll or download again immediately.
#             raise RuntimeError(
#                 f'Data setup paused for Kaggle login ({kind}{status_text}). '
#                 'Complete the login prompt above, then rerun THIS DATA CELL and continue the remaining cells. '
#                 'Do not restart the kernel after login. Existing data and valid caches need no login.') from None
#         if status == 403:
#             advice = ('Your account may need competition access or rule acceptance. '
#                       'If you need to switch credentials, use kagglehub.login() or set KAGGLE_API_TOKEN. '
#                       f'Confirm competition access and accepted rules at https://www.kaggle.com/competitions/{KAGGLE_SLUG}/rules.')
#         elif status == 404:
#             advice = f'Check the competition name {KAGGLE_SLUG!r} and that your account can access it.'
#         elif status == 429:
#             advice = 'Kaggle rate-limited the request. Wait before retrying.'
#         elif kind in ('ConnectionError', 'ConnectTimeout', 'ReadTimeout', 'Timeout'):
#             advice = 'Check the runtime internet connection and retry the data cell.'
#         elif isinstance(exc, OSError):
#             advice = 'Check local free space and write permissions for the KaggleHub cache.'
#         else:
#             advice = ('Check KaggleHub installation and account access; use kagglehub.login() or '
#                       'KAGGLE_API_TOKEN if authentication is required.')
#         # Keep useful diagnostics without echoing request URLs/tokens from exception text.
#         raise RuntimeError(f'KaggleHub download failed ({kind}{status_text}). {advice} '
#                            'Rerun the data cell after correcting the issue.') from None
#     if not downloaded.exists():
#         raise FileNotFoundError(f'KaggleHub returned a path that does not exist: {downloaded}')
#     print(f'[download] KaggleHub returned: {downloaded}')
#     return downloaded
#
#
# def download_competition_data(data_dir):
#     """Normalize a KaggleHub folder/ZIP or a legacy downloaded ZIP into the raw data directory."""
#     data_dir = Path(data_dir)
#     data_dir.parent.mkdir(parents=True, exist_ok=True)
#     legacy_archive = data_dir.parent / 'downloads' / KAGGLE_SLUG / f'{KAGGLE_SLUG}.zip'
#     downloaded = None
#     if legacy_archive.is_file():
#         try:
#             with zipfile.ZipFile(legacy_archive) as z:
#                 if z.testzip() is None:
#                     downloaded = legacy_archive
#         except (OSError, zipfile.BadZipFile, zlib.error, EOFError):
#             pass
#     if downloaded is None:
#         downloaded = run_kaggle_download()
#     else:
#         print(f'[download] reusing completed legacy archive: {downloaded}')
#     if downloaded.is_dir() and (downloaded.resolve().is_relative_to(data_dir.resolve())
#                                 or data_dir.resolve().is_relative_to(downloaded.resolve())):
#         raw = find_raw_data(downloaded, require_images=True)
#         if raw is not None:
#             return raw  # reuse overlapping complete data without moving or modifying the provider cache
#         raise ValueError('KaggleHub cache overlaps DATA_DIR. Choose a separate DATA_DIR for extraction.')
#     with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
#         extracted = Path(tmp) / 'extracted'
#         if downloaded.is_dir():
#             # KaggleHub normally already extracted the competition archive. Work on a copy:
#             # its cache (or Kaggle's read-only input mount) must not be moved or unpacked in place.
#             shutil.copytree(downloaded, extracted, copy_function=shutil.copyfile)
#             for directory, _, _ in os.walk(extracted):
#                 Path(directory).chmod(0o700)  # only the working copy must allow nested ZIP extraction
#         elif zipfile.is_zipfile(downloaded):
#             extract_zip_checked(downloaded, extracted)
#         else:
#             raise ValueError(f'Expected a competition folder or ZIP from KaggleHub, received {downloaded}')
#         unpacked = set()
#         for _ in range(4):
#             raw = find_raw_data(extracted, require_images=True)
#             if raw is not None:
#                 publish_data_directory(raw, data_dir)
#                 print(f'[data] ready: {data_dir}')
#                 return data_dir
#             nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
#             if not nested:
#                 break
#             for path in nested:
#                 target = path.parent
#                 if path.stem in ('train_images', 'test_images'):
#                     with zipfile.ZipFile(path) as z:
#                         if not any(name.startswith(path.stem + '/') for name in z.namelist()):
#                             target = target / path.stem
#                 extract_zip_checked(path, target)
#                 unpacked.add(path)
#     raise ValueError(f'Downloaded files at {downloaded} lack complete train_images/, test_images/ and '
#                      'the three CSVs. Inspect that folder or set DATA_DIR to complete extracted data. '
#                      'If the KaggleHub cache is incomplete, refresh it with '
#                      'kagglehub.competition_download(..., force_download=True).')
#
#

def prepare_uploaded_data(uploaded, data_dir):
    """Read a user-supplied folder/ZIP; unpack only into local staging, without network access."""
    uploaded, data_dir = Path(uploaded), Path(data_dir)
    if not uploaded.exists():
        raise FileNotFoundError(
            f'No complete data or valid cache. Upload the competition ZIP or extracted files to '
            f'{uploaded}. On Colab the default is MyDrive/DL_Project1/data/. '
            'Kaggle downloads and login are disabled.')
    if (uploaded.resolve().is_relative_to(data_dir.resolve())
            or data_dir.resolve().is_relative_to(uploaded.resolve())):
        raise ValueError('Uploaded source and local extraction directory must be separate.')
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f'[data] reading manually uploaded files: {uploaded}')
    with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
        extracted = Path(tmp) / 'extracted'
        if uploaded.is_dir():
            shutil.copytree(uploaded, extracted, copy_function=shutil.copyfile)
            for directory, _, _ in os.walk(extracted):
                Path(directory).chmod(0o700)
        elif zipfile.is_zipfile(uploaded):
            extract_zip_checked(uploaded, extracted)
        else:
            raise ValueError(f'Expected an uploaded competition folder or ZIP, received {uploaded}')
        unpacked = set()
        for _ in range(4):
            raw = find_raw_data(extracted, require_images=True)
            if raw is not None:
                publish_data_directory(raw, data_dir)
                print(f'[data] ready: {data_dir}')
                return data_dir
            nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
            if not nested:
                break
            for path in nested:
                target = path.parent
                if path.stem in ('train_images', 'test_images'):
                    with zipfile.ZipFile(path) as z:
                        if not any(name.startswith(path.stem + '/') for name in z.namelist()):
                            target = target / path.stem
                extract_zip_checked(path, target)
                unpacked.add(path)
    raise ValueError(
        f'Uploaded files at {uploaded} lack complete train_images/, test_images/, '
        'train_labels.csv, test.csv and sample_submission.csv. Upload all competition files; '
        'ZIP and extracted-folder layouts are supported.')


def prepare_data(paths, auto_download=False, save_to_drive=True):
    """Valid local cache -> compatible Drive cache -> manually uploaded folder/ZIP."""
    if auto_download:
        raise ValueError("Kaggle download/login is disabled. Keep AUTO_DOWNLOAD=False and upload data manually.")
    cache = paths.cache_dir
    persist = paths.env == 'colab' and save_to_drive
    if persist and not drive_mounted():
        raise RuntimeError('Mount Google Drive before data setup to save cache.zip, or explicitly set '
                           'SAVE_CACHE_TO_DRIVE=False for temporary storage only.')
    source = find_raw_data(paths.data_dir) or paths.data_dir
    meta = None
    if (cache / 'meta.json').is_file():
        try:
            candidate = verify_cache(cache)
            if _cache_matches_source(candidate, source):
                meta = candidate
                print(f'[cache] reusing verified local cache: {cache}')
            else:
                print('[cache] source CSVs changed; rebuilding from the requested data')
        except (OSError, ValueError, KeyError, RuntimeError, EOFError) as exc:
            print(f'[cache] local cache is incomplete or invalid ({type(exc).__name__})')
    if meta is None and paths.env == 'colab' and drive_mounted():
        try:
            fetch_drive_cache(drive_dir(), cache, source)
            meta = verify_cache(cache)
        except FileNotFoundError:
            pass
    if meta is None:
        if paths.data_dir.resolve() == cache.resolve():
            raise ValueError('Use separate DATA_DIR and CACHE_DIR when building from uploaded data')
        raw = find_raw_data(paths.data_dir, require_images=True)
        if raw is None:
            raw = prepare_uploaded_data(paths.data_dir, cache.parent / 'data_extracted')
        paths.data_dir = raw
        cache.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='.build-cache-', dir=cache.parent) as tmp:
            staged = Path(tmp) / 'cache'
            build_cache(raw, staged)
            meta = verify_cache(staged)
            publish_data_directory(staged, cache)
    else:
        paths.data_dir = source
    if persist:
        save_drive_cache(cache, drive_dir())
    return cache, meta


In [22]:
CACHE, META = prepare_data(PATHS, auto_download=AUTO_DOWNLOAD, save_to_drive=SAVE_CACHE_TO_DRIVE)
# Keep later training/evaluation cells on the resolved source folder (including wrapper directories).
DATA_DIR = str(PATHS.data_dir)
CACHE_DIR = str(CACHE)
print(f"[cache] {CACHE}: format {META['format']} | {META['n_train']:,} train / {META['n_test']:,} test images "
      f"{tuple(META['shape'])} | fingerprint {META['fingerprint'][:12]} | sha256 verified")
counts = np.bincount(np.load(CACHE / "train_labels.npy"), minlength=NUM_CLASSES)
print("Training images per class: " + ", ".join(f"{c} {n:,}" for c, n in zip(CLASSES, counts)))

[cache] reusing verified local cache: /content/cache
[cache] /content/cache: format 2 | 50,000 train / 2,000 test images (32, 32, 3) | fingerprint 42749a486a13 | sha256 verified
Training images per class: airplane 5,000, automobile 5,000, bird 5,000, cat 5,000, deer 5,000, dog 5,000, frog 5,000, horse 5,000, ship 5,000, truck 5,000


In [23]:
# ---- split and normalization statistics

def _per_class(n, what):
    if not isinstance(n, (int, np.integer)) or isinstance(n, (bool, np.bool_)) or n < 0:
        raise ValueError(f"{what} must be a nonnegative integer, got {n!r}")
    if n % NUM_CLASSES:
        raise ValueError(f"{what} must be a multiple of {NUM_CLASSES} for a class-balanced split, got {n}")
    return n // NUM_CLASSES


def stratified_split(labels, val_size, seed=0):
    """Hold out `val_size` images, the same number from every class. Returns sorted (train_idx, val_idx)."""
    labels = np.asarray(labels)
    if labels.ndim != 1 or not np.issubdtype(labels.dtype, np.integer) or not len(labels):
        raise ValueError("labels must be a nonempty one-dimensional integer array")
    if np.any((labels < 0) | (labels >= NUM_CLASSES)):
        raise ValueError("labels must be in 0..9")
    k = _per_class(val_size, "val_size")
    if val_size >= len(labels):
        raise ValueError("Validation size must leave at least one training image")
    if np.any(np.bincount(labels, minlength=NUM_CLASSES) < k):
        raise ValueError("Not enough examples per class for the requested validation split")
    all_idx = np.arange(len(labels))
    if val_size == 0:
        return all_idx, np.array([], dtype=np.int64)
    k = _per_class(val_size, "val_size")
    rng = np.random.default_rng(seed)
    val = np.concatenate([rng.permutation(np.flatnonzero(labels == c))[:k] for c in range(NUM_CLASSES)])
    val = np.sort(val)
    return np.setdiff1d(all_idx, val), val


def stratified_subset(indices, labels, size, seed=0):
    """Class-balanced subset of `indices` (used for quick smoke tests)."""
    k = _per_class(size, "train_subset")
    if size == 0 or size > len(indices) or np.any(np.bincount(labels[indices], minlength=NUM_CLASSES) < k):
        raise ValueError("Subset size must be positive and available in every training class")
    rng = np.random.default_rng(seed + 1)
    sub = [rng.permutation(indices[labels[indices] == c])[:k] for c in range(NUM_CLASSES)]
    return np.sort(np.concatenate(sub))


def channel_stats(images):
    """Per-channel mean/std in [0, 1] units over uint8 NHWC images (computed in chunks)."""
    s = np.zeros(3)
    ss = np.zeros(3)
    n = 0
    for i in range(0, len(images), 5000):
        chunk = np.asarray(images[i:i + 5000], dtype=np.float64) / 255.0
        s += chunk.sum(axis=(0, 1, 2))
        ss += (chunk ** 2).sum(axis=(0, 1, 2))
        n += chunk.shape[0] * chunk.shape[1] * chunk.shape[2]
    mean = s / n
    std = np.sqrt(ss / n - mean ** 2)
    return mean.tolist(), std.tolist()


# ---- batch augmentation on the accelerator

def _to_dev(t, dev):
    """Copy a small CPU tensor to `dev`; on CUDA via pinned memory without a host sync."""
    if dev.type == "cuda":
        return t.pin_memory().to(dev, non_blocking=True)
    return t.to(dev)


def epoch_generator(seed, epoch, stream):
    """CPU generator for a given (seed, epoch, stream); device independent and resumable."""
    state = np.random.SeedSequence([seed, epoch, stream]).generate_state(1, dtype=np.uint64)[0]
    return torch.Generator().manual_seed(int(state))


def to_float_nchw(x_u8, mean, std, channels_last=False):
    """uint8 NHWC on device -> normalized float NCHW (channels_last strides are free from NHWC)."""
    x = x_u8.permute(0, 3, 1, 2).float().div_(255.0)
    x = (x - mean) / std
    return x.contiguous(memory_format=torch.channels_last if channels_last else torch.contiguous_format)


def crop_pad(x_u8, pad=4, padding_mode="constant"):
    """Pad uint8 NHWC images before random cropping; reflect excludes the edge pixel (same code as scheme C/T31)."""
    if padding_mode not in ("constant", "reflect"):
        raise ValueError("crop_padding_mode must be constant or reflect")
    if padding_mode == "constant":
        return F.pad(x_u8, (0, 0, pad, pad, pad, pad))  # original black-padding path
    h, w = x_u8.shape[1:3]
    if not isinstance(pad, int) or not 0 <= pad < min(h, w):
        raise ValueError("Reflection padding must be nonnegative and smaller than the image")
    # Integer indexing preserves uint8 on both CPU and CUDA, with no random draws.
    rows = torch.arange(-pad, h + pad, device=x_u8.device)
    cols = torch.arange(-pad, w + pad, device=x_u8.device)
    rows = torch.where(rows < 0, -rows, torch.where(rows >= h, 2 * h - 2 - rows, rows))
    cols = torch.where(cols < 0, -cols, torch.where(cols >= w, 2 * w - 2 - cols, cols))
    return x_u8.index_select(1, rows).index_select(2, cols)


def random_crop_flip(x_u8, gen, pad=4, padding_mode="constant"):
    """Per-sample random crop with the selected padding plus horizontal flip on a uint8 NHWC batch."""
    b, h, w, _ = x_u8.shape
    dev = x_u8.device
    padded = crop_pad(x_u8, pad, padding_mode)
    offs = _to_dev(torch.randint(0, 2 * pad + 1, (b, 2), generator=gen), dev)
    flip = _to_dev(torch.rand(b, generator=gen) < 0.5, dev)
    rows = offs[:, 0, None] + torch.arange(h, device=dev)
    cols = torch.arange(w, device=dev).expand(b, w)
    cols = torch.where(flip[:, None], cols.flip(1), cols) + offs[:, 1, None]
    bidx = torch.arange(b, device=dev)[:, None, None]
    return padded[bidx, rows[:, :, None], cols[:, None, :]]


def cutout(x, gen, size=16):
    """Zero a size x size square per sample, center uniform over the image (DeVries & Taylor, 2017).

    Applied after normalization, so zero equals the training-set mean colour, as in the paper's code.
    """
    b, _, h, w = x.shape
    dev = x.device
    cy = _to_dev(torch.randint(0, h, (b,), generator=gen), dev)
    cx = _to_dev(torch.randint(0, w, (b,), generator=gen), dev)
    ys = torch.arange(h, device=dev)
    xs = torch.arange(w, device=dev)
    in_y = (ys[None] >= (cy - size // 2)[:, None]) & (ys[None] < (cy + size - size // 2)[:, None])
    in_x = (xs[None] >= (cx - size // 2)[:, None]) & (xs[None] < (cx + size - size // 2)[:, None])
    mask = ~(in_y[:, :, None] & in_x[:, None, :])
    return x * mask[:, None].to(x.dtype)


def random_erasing(x, gen, p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), attempts=10):
    """Per-sample RandomErasing (Zhong et al., 2020) with value 0, same sampling as torchvision."""
    b, _, h, w = x.shape
    area = h * w
    target = area * torch.empty(b, attempts).uniform_(scale[0], scale[1], generator=gen)
    log_r = torch.empty(b, attempts).uniform_(np.log(ratio[0]), np.log(ratio[1]), generator=gen)
    r = torch.exp(log_r)
    eh = torch.sqrt(target * r).round().long()
    ew = torch.sqrt(target / r).round().long()
    valid = (eh < h) & (ew < w)
    first = valid.float().argmax(1)  # first attempt that fits (torchvision tries up to 10)
    pick = torch.arange(b)
    eh, ew = eh[pick, first], ew[pick, first]
    apply = valid.any(1) & (torch.rand(b, generator=gen) < p)
    top = (torch.rand(b, generator=gen) * (h - eh + 1)).long()
    left = (torch.rand(b, generator=gen) * (w - ew + 1)).long()
    params = _to_dev(torch.stack([top, left, eh, ew, apply.long()], 1), x.device)
    ys = torch.arange(h, device=x.device)[None]
    xs = torch.arange(w, device=x.device)[None]
    t, l, hh, ww, a = params.unbind(1)
    in_y = (ys >= t[:, None]) & (ys < (t + hh)[:, None])
    in_x = (xs >= l[:, None]) & (xs < (l + ww)[:, None])
    mask = ~(in_y[:, :, None] & in_x[:, None, :] & a.bool()[:, None, None])
    return x * mask[:, None].to(x.dtype)


# ---- AutoAugment / TrivialAugment, one image at a time on the CPU

# AUG values whose policy runs per image on the CPU; AUTOAUG_ERASE then chooses the occlusion after normalization.
CPU_POLICY_AUGS = ("autoaug", "trivialaug")


class AutoAugmentDataset(torch.utils.data.Dataset):
    """Applies RandomCrop(32, pad 4) + flip + a per-image policy to one uint8 image at a time.

    `policy` "autoaug" is AutoAugment (CIFAR10 policy); "trivialaug" is TrivialAugmentWide (Mueller & Hutter, 2021)
    with torchvision's defaults (31 magnitude bins, nearest interpolation, fill 0), the same defaults as AutoAugment.
    Indices arrive as `epoch * len(self) + position` from `EpochSampler`; the per-image RNG is seeded
    from that number, so results do not depend on which worker handles which image.
    """

    def __init__(self, images_path, indices, labels, seed, policy="autoaug", padding_mode="constant"):
        from torchvision.transforms import v2

        if policy not in CPU_POLICY_AUGS:
            raise ValueError(f"policy = {policy!r}; choose one of {CPU_POLICY_AUGS}")
        self.images_path = str(images_path)
        self.indices = np.asarray(indices)
        self.labels = torch.as_tensor(labels[self.indices], dtype=torch.long)
        self.seed = int(seed)
        self.policy = policy
        self.images = None  # opened lazily as a memmap inside each worker
        if padding_mode not in ("constant", "reflect"):
            raise ValueError("crop_padding_mode must be constant or reflect")
        self.padding_mode = padding_mode
        self.transform = v2.Compose([
            v2.RandomCrop(32, padding=4, padding_mode=padding_mode),
            v2.RandomHorizontalFlip(),
            v2.AutoAugment(v2.AutoAugmentPolicy.CIFAR10) if policy == "autoaug" else v2.TrivialAugmentWide(),
        ])

    def describe(self):
        """The installed policy and its settings, as recorded in the run configuration."""
        op = self.transform.transforms[-1]
        return {"policy": self.policy, "crop_padding_mode": self.padding_mode,
                "transform": repr(self.transform), "operation": type(op).__name__,
                "num_magnitude_bins": getattr(op, "num_magnitude_bins", None),
                "interpolation": str(op.interpolation.value), "fill": op.fill,
                "order": "uint8 crop/flip -> policy (CPU) -> float/normalize -> AUTOAUG_ERASE occlusion"}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, g):
        if self.images is None:
            self.images = np.load(self.images_path, mmap_mode="r")
        pos = g % len(self.indices)
        torch.manual_seed((self.seed * 10**10 + g) % 2**63)
        img = torch.from_numpy(np.array(self.images[self.indices[pos]])).permute(2, 0, 1)
        return self.transform(img).permute(1, 2, 0).contiguous(), self.labels[pos]  # back to HWC


class EpochSampler(torch.utils.data.Sampler):
    """Shuffles positions with a (seed, epoch) generator and encodes the epoch into each index."""

    def __init__(self, n, seed):
        self.n, self.seed, self.epoch = n, seed, 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        perm = torch.randperm(self.n, generator=epoch_generator(self.seed, self.epoch, 0))
        return iter((perm + self.epoch * self.n).tolist())

    def __len__(self):
        return self.n


# ---- training batches for one epoch

class TrainBatches:
    """Yields normalized, augmented (x, y) training batches on `device` for one epoch at a time."""

    def __init__(self, images, labels, train_idx, args, mean, std, device, images_path=None):
        self.args, self.device = args, device
        self.mean = torch.tensor(mean, device=device).view(1, 3, 1, 1)
        self.std = torch.tensor(std, device=device).view(1, 3, 1, 1)
        self.n = len(train_idx)
        self.batch_size = args.batch_size
        if self.batch_size <= 0 or self.n < self.batch_size:
            raise ValueError(f"Need at least one complete training batch: {self.n} images, batch {self.batch_size}")
        self.steps_per_epoch = self.n // self.batch_size  # drop the last partial batch
        self.loader = None
        if args.aug in CPU_POLICY_AUGS:
            ds = AutoAugmentDataset(images_path, train_idx, labels, args.seed, policy=args.aug,
                                    padding_mode=getattr(args, "crop_padding_mode", "constant"))
            self.sampler = EpochSampler(self.n, args.seed)
            batch_sampler = torch.utils.data.BatchSampler(self.sampler, self.batch_size, drop_last=True)
            kw = dict(num_workers=args.workers, pin_memory=bool(getattr(args, "pin_memory", device.type == "cuda")))
            if args.workers > 0:
                kw.update(persistent_workers=True, prefetch_factor=4, multiprocessing_context=MP_CONTEXT)
            else:
                name = "AutoAugment" if args.aug == "autoaug" else "TrivialAugment"
                print(f"{name} runs in the main process (workers = 0; see section 2)")
            self.loader = torch.utils.data.DataLoader(ds, batch_sampler=batch_sampler, **kw)
        else:
            self.x = torch.from_numpy(np.ascontiguousarray(images[train_idx])).to(device)
            self.y = torch.from_numpy(labels[train_idx]).to(device)

    def _finish(self, x_u8, gen):
        a = self.args
        x = to_float_nchw(x_u8, self.mean, self.std, a.channels_last)
        if a.aug == "cutout" or (a.aug in CPU_POLICY_AUGS and a.autoaug_erase == "cutout"):
            x = cutout(x, gen, a.cutout_size)
        elif a.aug in CPU_POLICY_AUGS and a.autoaug_erase == "erasing":
            x = random_erasing(x, gen, p=a.erasing_p)
        if a.channels_last:  # keep NHWC strides after the masking ops (no-op if already)
            x = x.contiguous(memory_format=torch.channels_last)
        return x

    def epoch(self, epoch):
        seed = self.args.seed
        gen = epoch_generator(seed, epoch, 1)
        if self.loader is not None:
            self.sampler.set_epoch(epoch)
            for x_u8, y in self.loader:
                x_u8 = x_u8.to(self.device, non_blocking=True)
                yield self._finish(x_u8, gen), y.to(self.device, non_blocking=True)
            return
        perm = torch.randperm(self.n, generator=epoch_generator(seed, epoch, 0)).to(self.device)
        for i in range(self.steps_per_epoch):
            idx = perm[i * self.batch_size:(i + 1) * self.batch_size]
            x_u8 = (random_crop_flip(self.x[idx], gen,
                                     padding_mode=getattr(self.args, "crop_padding_mode", "constant"))
                    if self.args.aug in ("basic", "cutout") else self.x[idx])
            yield self._finish(x_u8, gen), self.y[idx]

In [24]:
MODEL = "E"
MODEL_DESCRIPTION = "PyramidNet-110 basic, alpha=98 (widths 16->114)"  # the default (E1/T18, T106) architecture
EXPECTED_PARAMS = 4_963_991  # default basic-110, alpha 98; other BLOCK / DEPTH / ALPHA values: expected_param_count()


class DropPath(nn.Module):
    """Stochastic depth (Huang et al., 2016) for one residual branch, per sample.

    In training, each image's branch is dropped with probability p and survivors are scaled by 1 / (1 - p);
    in evaluation it is the identity. The mask comes from the global torch RNG of the input's device.
    """

    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        keep = 1.0 - self.p
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep
        return x * mask.to(x.dtype) / keep

    def extra_repr(self):
        return f"p={self.p:.4f}"


class ShakeDrop(nn.Module):
    """ShakeDrop (Yamada, Iwamura, Akiba & Kise, 2019, arXiv 1802.02375) on one residual branch output F(x).

    p is the probability that this block is perturbed. Training: a gate b ~ Bernoulli(1 - p) per image, then
    alpha ~ U[-1, 1] and beta ~ U[0, 1] per element of F(x) (the paper's pixel-level rule). The forward pass
    multiplies F(x) by b + alpha - b*alpha, the backward pass multiplies the gradient by b + beta - b*beta: a kept
    branch (b = 1) is untouched, a perturbed one is scaled by alpha forward and by beta backward. Evaluation multiplies
    F(x) by E[b + alpha - b*alpha] = 1 - p. Like DropPath, the draws come from the default RNG of the input's device
    (gate, then alpha, then beta), inside the (compiled) training graph; no custom autograd.Function is needed:
    y = beta_coef * F + (alpha_coef - beta_coef) * F.detach() has the forward value alpha_coef * F and the gradient
    beta_coef. Coefficients and the product are computed in float32 and cast back to F's dtype (bf16 under autocast);
    channels_last is preserved. No parameters or buffers.
    """

    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)

    def coefficients(self, x):
        """(forward, backward) float32 coefficients for a training batch shaped like x (one gate per image).

        b + alpha - b*alpha is computed as b + (1 - b) * alpha, which is exactly 1 for a kept branch (b = 1) and exactly
        alpha for a perturbed one (b = 0); likewise for beta.
        """
        keep = 1.0 - self.p
        gate = (torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep).to(torch.float32)
        alpha = torch.rand_like(x, dtype=torch.float32) * 2.0 - 1.0  # U[-1, 1]
        beta = torch.rand_like(x, dtype=torch.float32)  # U[0, 1]
        perturbed = 1.0 - gate
        return gate + perturbed * alpha, gate + perturbed * beta

    def forward(self, x):
        if self.p == 0.0:
            return x
        if not self.training:
            return x * (1.0 - self.p)  # E[b + alpha - b*alpha] = keep probability
        forward, backward = self.coefficients(x)
        x32 = x.float()
        return (x32 * backward + x32.detach() * (forward - backward)).to(x.dtype)

    def extra_repr(self):
        return f"p={self.p:.4f}"


def sam_replay_needed(model):
    """True if the training forward draws ShakeDrop coefficients: SAM's second pass then replays the first pass's
    draws by restoring the device RNG (SharpnessAwareMinimization, section 5). DropPath masks stay independent."""
    return any(isinstance(m, ShakeDrop) and m.p > 0 for m in model.modules())


MIXSTYLE_AFTER_STAGES = (1, 2)  # training-only MixStyle (MIXSTYLE, section 1) mixes the outputs of stages 1 and 2


def mix_feature_styles(x, apply, lam, perm, eps):
    """MixStyle (Zhou et al., ICLR 2021), "random" variant, on one training batch; parameter-free.

    Each image's per-channel mean mu and standard deviation sigma over the spatial positions (unbiased variance + eps,
    computed in float32 and detached, as in the reference code) are replaced by lam * stat + (1 - lam) * stat of its
    partner image perm[i]; the result is cast back to x's dtype. `apply` (0-dim bool tensor) selects the mixed batch
    or exactly x; lam has shape (B, 1, 1, 1). The draws come from the trainer's MixStyleSampler (section 5).
    """
    x32 = x.float()
    mu = x32.mean(dim=(2, 3), keepdim=True).detach()
    sigma = (x32.var(dim=(2, 3), keepdim=True) + eps).sqrt().detach()
    mixed_mu = lam * mu + (1 - lam) * mu[perm]
    mixed_sigma = lam * sigma + (1 - lam) * sigma[perm]
    mixed = ((x32 - mu) / sigma * mixed_sigma + mixed_mu).to(x.dtype)
    return torch.where(apply, mixed, x)


class PyramidBlock(nn.Module):
    """BN-conv-BN-ReLU-conv-BN (Han et al., 2017), added to a parameter-free shortcut; no ReLU after the addition.

    The ShakeDrop module is registered only when its rate is positive, so the default block (and the T106 model
    fingerprint, which includes repr(model)) is unchanged.
    """

    def __init__(self, in_ch, out_ch, stride=1, drop_path=0.0, shakedrop=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path)
        self.shakedrop = ShakeDrop(shakedrop) if shakedrop else None
        # Shortcut: identity, 2x2 average pool where the block downsamples, zero padding for the extra channels.
        self.pool = nn.AvgPool2d(2, 2, ceil_mode=True) if stride != 1 else nn.Identity()
        self.extra_ch = out_ch - in_ch

    def forward(self, x):
        out = self.conv1(self.bn1(x))
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.drop_path(self.bn3(out))
        if self.shakedrop is not None:  # F(x) before the shortcut addition
            out = self.shakedrop(out)
        shortcut = self.pool(x)
        if self.extra_ch:
            shortcut = F.pad(shortcut, (0, 0, 0, 0, 0, self.extra_ch))
        return out + shortcut


class PyramidBottleneck(nn.Module):
    """Bottleneck PyramidNet block (Han et al., 2017; dyhan0920/PyramidNet-PyTorch): BN-conv1x1-BN-ReLU-conv3x3
    (stride)-BN-ReLU-conv1x1(x4)-BN, added to the same parameter-free shortcut (2x2 average pool where the block
    downsamples, zero-padded channels); no ReLU after the addition. `planes` is the pyramid width; the block outputs
    4 * planes channels."""

    expansion = 4

    def __init__(self, in_ch, planes, stride=1, drop_path=0.0, shakedrop=0.0):
        super().__init__()
        out_ch = planes * self.expansion
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, planes, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, out_ch, 1, bias=False)
        self.bn4 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path)
        self.shakedrop = ShakeDrop(shakedrop) if shakedrop else None
        self.pool = nn.AvgPool2d(2, 2, ceil_mode=True) if stride != 1 else nn.Identity()
        self.extra_ch = out_ch - in_ch

    def forward(self, x):
        out = self.conv1(self.bn1(x))
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.conv3(F.relu(self.bn3(out)))
        out = self.drop_path(self.bn4(out))
        if self.shakedrop is not None:
            out = self.shakedrop(out)
        shortcut = self.pool(x)
        if self.extra_ch:
            shortcut = F.pad(shortcut, (0, 0, 0, 0, 0, self.extra_ch))
        return out + shortcut


# BLOCK -> (layers per block, so depth = 3 * layers * n + 2, and the block's output channels per pyramid width)
PYRAMID_BLOCKS = {"basic": (2, 1), "bottleneck": (3, PyramidBottleneck.expansion)}


def blocks_per_stage(depth, block):
    """n blocks per stage: (depth - 2) / 6 for basic blocks, (depth - 2) / 9 for bottlenecks."""
    if block not in PYRAMID_BLOCKS:
        raise ValueError(f"BLOCK = {block!r}; choose one of {tuple(PYRAMID_BLOCKS)}")
    per_stage = 3 * PYRAMID_BLOCKS[block][0]
    if type(depth) is not int or depth < per_stage + 2 or (depth - 2) % per_stage:
        raise ValueError(f"DEPTH = {depth!r} is invalid for {block} blocks: use {per_stage}n + 2 with n >= 1 "
                         f"(e.g. {'110' if block == 'basic' else '164, 200 or 272'})")
    return (depth - 2) // per_stage


class PyramidNet(nn.Module):
    """3x3 stem (16 channels, BN), 3 stages of n pyramidal blocks, BN-ReLU, global average pool, linear.

    n = (depth-2)/6 basic or (depth-2)/9 bottleneck blocks per stage. The width grows by alpha / N after each of the
    N = 3n blocks. As in the reference implementation, the width is accumulated as a float and each block uses
    int(round(width)) channels (bottlenecks output 4x that). Block l of N drops its residual branch with probability
    drop_path * l / N (linear decay rule of Huang et al., 2016), or, with ShakeDrop, is perturbed with probability
    shakedrop * l / N (Yamada et al.'s linear rule with p_L = 1 - shakedrop); the two are exclusive. A training forward
    may receive MixStyle draws for the points after stages MIXSTYLE_AFTER_STAGES; this adds no module, parameter or
    buffer.
    """

    mixstyle_layers = len(MIXSTYLE_AFTER_STAGES)  # number of MixStyle draws a training forward accepts

    def __init__(self, depth=110, alpha=98, drop_path=0.0, num_classes=10, block="basic", shakedrop=0.0):
        super().__init__()
        if drop_path and shakedrop:
            raise ValueError("DropPath and ShakeDrop act on the same residual branch; enable only one of them")
        n = blocks_per_stage(depth, block)
        unit = PyramidBlock if block == "basic" else PyramidBottleneck
        expansion = PYRAMID_BLOCKS[block][1]
        total = 3 * n
        add_rate = alpha / total
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16))
        stages, in_ch, width, index = [], 16, 16.0, 0
        for i in range(3):
            layers = []
            for j in range(n):
                index += 1
                width += add_rate
                planes = int(round(width))
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(unit(in_ch, planes, stride, drop_path * index / total, shakedrop * index / total))
                in_ch = planes * expansion
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.bn = nn.BatchNorm2d(in_ch)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x, mixstyle=None):
        """`mixstyle` = (apply, lam, perm, eps) from MixStyleSampler.draw, or None. Without draws, and always in
        evaluation mode, this is exactly the plain PyramidNet forward."""
        x = self.stem(x)
        if mixstyle is None or not self.training:
            x = self.stages(x)
        else:
            apply, lam, perm, eps = mixstyle
            layer = 0
            for index, stage in enumerate(self.stages, 1):
                x = stage(x)
                if index in MIXSTYLE_AFTER_STAGES:
                    x = mix_feature_styles(x, apply[layer], lam[layer], perm[layer], eps)
                    layer += 1
        x = F.relu(self.bn(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


def pyramid_widths(depth=110, alpha=98, block="basic"):
    """Pyramid width of every block and output channels of each stage, by PyramidNet's int(round(width)) rule."""
    n = blocks_per_stage(depth, block)
    rate = alpha / (3 * n)
    width, planes = 16.0, []
    for _ in range(3 * n):
        width += rate
        planes.append(int(round(width)))
    expansion = PYRAMID_BLOCKS[block][1]
    return planes, [planes[(i + 1) * n - 1] * expansion for i in range(3)]


# Always-present options (unchanged since E1) and optional ones that are omitted from the saved options when at their
# default, so the E1/T18 and T106 options, descriptions and model fingerprints are reproduced exactly.
ARCH_DEFAULTS = {"implementation_version": 1, "drop_path": 0.0}
ARCH_OPTION_DEFAULTS = {"shakedrop": 0.0, "block": "basic", "depth": 110, "alpha": 98}


def arch_option(options, key):
    return options.get(key, ARCH_OPTION_DEFAULTS[key])


def architecture_from_config():
    return {"drop_path": DROP_PATH, "shakedrop": SHAKEDROP, "block": BLOCK, "depth": DEPTH, "alpha": ALPHA}


def validate_architecture(options):
    """Check the resolved options in place (SHAKEDROP becomes a float, an integral float ALPHA an int)."""
    rate = options["drop_path"]
    if isinstance(rate, bool) or not isinstance(rate, (int, float)) or not 0 <= rate < 1:
        raise ValueError(f"DROP_PATH = {rate!r}; choose a rate in [0, 1) (0 = no stochastic depth)")
    shake = options["shakedrop"]
    if isinstance(shake, bool) or not isinstance(shake, (int, float)) or not 0 <= shake < 1:
        raise ValueError(f"SHAKEDROP = {shake!r}; choose a rate in [0, 1) (0 = off; 0.5 = the paper's p_L = 0.5)")
    options["shakedrop"] = float(shake)
    if rate > 0 and shake > 0:
        raise ValueError(f"DROP_PATH = {rate!r} and SHAKEDROP = {shake!r}: both act on the same residual branch; "
                         "set one of them to 0")
    blocks_per_stage(options["depth"], options["block"])  # checks BLOCK and DEPTH
    alpha = options["alpha"]
    if isinstance(alpha, bool) or not isinstance(alpha, (int, float)) or not math.isfinite(alpha) or alpha <= 0:
        raise ValueError(f"ALPHA = {alpha!r}; choose a finite positive widening (98 for basic-110)")
    if isinstance(alpha, float) and alpha.is_integer():
        options["alpha"] = int(alpha)  # 100.0 and 100 are the same architecture and fingerprint
    count = expected_param_count(options)
    if count > PARAM_BUDGET:
        raise ValueError(f"{options['block']} PyramidNet-{options['depth']} with alpha {options['alpha']} has "
                         f"{count:,} parameters, over the {PARAM_BUDGET:,} budget")


def describe_architecture(options):
    # The rates are saved with the options and enter the architecture fingerprint: resuming with another rate is
    # rejected, and evaluation rebuilds the saved options whatever the current switches. For the default options the
    # first part is exactly MODEL_DESCRIPTION.
    block, depth, alpha = arch_option(options, "block"), arch_option(options, "depth"), arch_option(options, "alpha")
    planes, ends = pyramid_widths(depth, alpha, block)
    text = (f"PyramidNet-{depth} {block}, alpha={alpha} (widths 16->{planes[-1]}"
            + (f", block outputs {planes[0] * PYRAMID_BLOCKS[block][1]}->{ends[-1]}" if block != "basic" else "")
            + f"), stochastic depth DROP_PATH={float(options['drop_path'])!r}")
    shake = arch_option(options, "shakedrop")
    if shake:
        total = 3 * blocks_per_stage(depth, block)
        text += (f", ShakeDrop SHAKEDROP={float(shake)!r} (block l of L={total} perturbed with probability "
                 f"{float(shake)!r}*l/L, p_L={1 - shake:g}; gate per image, alpha/beta per element)")
    return text


def architecture_feature(options):
    if arch_option(options, "block") != "basic":
        return "bottleneck"
    if arch_option(options, "shakedrop") > 0:
        return "shakedrop"
    return "stochdepth" if options["drop_path"] > 0 else None


def expected_param_count(options):
    """Exact trainable parameters from the widths (bias-free convolutions, BN gamma/beta, linear weight and bias).
    DropPath, ShakeDrop and MixStyle add none."""
    block, depth, alpha = arch_option(options, "block"), arch_option(options, "depth"), arch_option(options, "alpha")
    planes, _ = pyramid_widths(depth, alpha, block)
    expansion = PYRAMID_BLOCKS[block][1]
    total, in_ch = 3 * 16 * 9 + 2 * 16, 16  # stem conv + BN
    for p in planes:
        out = p * expansion
        if block == "basic":
            total += 2 * in_ch + 9 * in_ch * p + 2 * p + 9 * p * p + 2 * p
        else:
            total += 2 * in_ch + in_ch * p + 2 * p + 9 * p * p + 2 * p + p * out + 2 * out
        in_ch = out
    return total + 2 * in_ch + in_ch * NUM_CLASSES + NUM_CLASSES  # final BN + linear


def architecture_topology(options):
    """Stem width, output channels of each stage (the width grows by alpha/N in every block) and blocks per stage."""
    block, depth, alpha = arch_option(options, "block"), arch_option(options, "depth"), arch_option(options, "alpha")
    n = blocks_per_stage(depth, block)
    return {"stem_channels": 16, "stage_channels": pyramid_widths(depth, alpha, block)[1], "blocks": [n, n, n]}


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return PyramidNet(depth=arch_option(options, "depth"), alpha=arch_option(options, "alpha"),
                      drop_path=options["drop_path"], num_classes=num_classes, block=arch_option(options, "block"),
                      shakedrop=arch_option(options, "shakedrop"))

In [25]:
PARAM_BUDGET = 5_000_000

def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model

def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

ARCHITECTURE_IMPLEMENTATION_VERSION = 1

def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - (set(ARCH_DEFAULTS) | set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = {**ARCH_DEFAULTS, **ARCH_OPTION_DEFAULTS, **options}
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Canonical omission: default-valued optional options (SHAKEDROP 0, basic blocks, depth 110, alpha 98) are dropped,
    # so E1/T18 and T106 options, architecture records and fingerprints are reproduced exactly.
    return {k: v for k, v in resolved.items() if k not in ARCH_OPTION_DEFAULTS or v != ARCH_OPTION_DEFAULTS[k]}

def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)

def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)

def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


In [26]:
def load_checkpoint(path):
    state = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(state, dict) or "args" not in state:
        return state
    original = state["args"].get("run_name")
    canonical = canonical_run_id(original)
    if original != canonical:
        state.setdefault("original_identity", {"run_id": original,
                         "model_version": state["args"].get("model_version")})
        state["args"] = dict(state["args"], run_name=canonical,
                             model_version=int(re.search(r"-ver-(\d+)-", canonical).group(1)),
                             task_id=run_task_id(canonical))
    return state

def predict_logits(model, images_u8, mean, std, device, batch_size=1000, flip=False, channels_last=False):
    """Logits for uint8 NHWC images (numpy or tensor); returns a CPU float tensor.

    With flip=True, returns the logits of horizontally flipped images instead.
    """
    model.eval()
    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
    out = []
    for i in range(0, len(images_u8), batch_size):
        xb = torch.as_tensor(np.asarray(images_u8[i:i + batch_size])).to(device)
        x = to_float_nchw(xb, mean_t, std_t, channels_last)
        if flip:
            x = x.flip(3)
        out.append(model(x).float().cpu())
    return torch.cat(out)

CHECKPOINT_FORMAT = 3

def model_fingerprint(model):
    options = getattr(model, "_project_architecture", ARCH_DEFAULTS)
    definition = getattr(model, "_project_description", MODEL_DESCRIPTION) + "\n" + repr(model)
    # Preserve the exact legacy baseline hash. New variants also bind options and implementation version.
    if options != ARCH_DEFAULTS:
        definition += "\n" + json.dumps(options, sort_keys=True)
    return hashlib.sha256(definition.encode()).hexdigest()

def validate_checkpoint(ckpt, fingerprint, expected_model_hash=None, run_id=None):
    if ckpt.get("format") != CHECKPOINT_FORMAT or "cache_fingerprint" not in ckpt:
        raise ValueError("Legacy checkpoint lacks verified dataset identity; start a fresh run with this notebook")
    if ckpt["cache_fingerprint"] != fingerprint:
        raise ValueError("Checkpoint dataset fingerprint differs from the current cache; refusing resume/evaluation")
    if ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint belongs to another model")
    saved_options = saved_architecture(ckpt["args"])
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(saved_options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if expected_model_hash is not None and ckpt.get("model_fingerprint") != expected_model_hash:
        raise ValueError("Checkpoint architecture differs from this notebook")
    if run_id is not None and canonical_run_id(ckpt["args"]["run_name"]) != canonical_run_id(run_id):
        raise ValueError("Checkpoint belongs to another run ID")


In [27]:
def load_model(ckpt, weights, device):
    options = saved_architecture(ckpt["args"])
    model, n_params = build_model(architecture=options)
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if "n_params" in ckpt and ckpt["n_params"] != n_params:
        raise ValueError("Checkpoint parameter count differs from its saved architecture")
    if ckpt.get("model_fingerprint") != model_fingerprint(model) or ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint architecture differs from this notebook")
    model.load_state_dict(ckpt["ema"]["module"] if weights == "ema" else ckpt["model"])
    return model.to(device).eval(), n_params

def validate_submission(path, sample_path):
    sub, sample = pd.read_csv(path), pd.read_csv(sample_path)
    if list(sub.columns) != ["id", "label"] or len(sub) != len(sample):
        raise ValueError("Submission columns or row count differ from the sample")
    if not sub["id"].is_unique or not np.array_equal(sub["id"], sample["id"]):
        raise ValueError("Submission IDs are duplicated or differ from sample order")
    if not pd.api.types.is_integer_dtype(sub["label"]) or not sub["label"].between(0,9).all():
        raise ValueError("Submission labels must be integers in 0..9")
    return len(sub)


In [28]:
import gc
ORIGINAL_RUN = "11-04-01-58_E_pyramid-ver-33-T128-asam-t122reg"
EXPECTED_CP_HASHES = {'11-04-01-58_E_pyramid-ver-33-T128-asam-t122reg_best-ckpt.pt': '4efba0062a2b3ab118e6f1058a6ac69d708221cbc7e084c767949db0eee56054', '11-04-01-58_E_pyramid-ver-33-T128-asam-t122reg_last-ckpt.pt': '969bbb16403f6340733f5d363089ec3721cd3bffc09ef4c97d4f6eb0aa855301'}
checkpoint_dir = PATHS.result_dir / "checkpoints" / ORIGINAL_RUN
output_dir = PATHS.result_dir / "evaluations" / "test" / ORIGINAL_RUN
assert output_dir.is_dir(), output_dir
checkpoint_files = sorted(checkpoint_dir.glob("*.pt"))
assert {p.name for p in checkpoint_files} == set(EXPECTED_CP_HASHES), "Checkpoint inventory changed; inspect before export"
print("Saved checkpoint files:", [p.name for p in checkpoint_files], flush=True)
images, ids = load_test(PATHS.data_dir, CACHE)
sample = pd.read_csv(CACHE / "sample_submission.csv")
assert len(ids) == len(sample) == 2000 and len(set(ids)) == 2000
row_of = {int(x):i for i,x in enumerate(ids)}
assert set(sample["id"]) == set(row_of)
order = np.array([row_of[int(x)] for x in sample["id"]])
stamp = run_timestamp()
identity = ORIGINAL_RUN.split("_",1)[1]
created=[]; reports=[]
for cp in checkpoint_files:
    cp_hash = sha256_file(cp)
    assert cp_hash == EXPECTED_CP_HASHES[cp.name], "Checkpoint hash differs from completed-run manifest"
    ck = load_checkpoint(cp)
    validate_checkpoint(ck, META["fingerprint"], run_id=ORIGINAL_RUN)
    role = "best" if "_best-ckpt" in cp.name else "last"
    assert ck["epoch"] == {"best":848,"last":900}[role]
    assert ck["ema"] is not None
    for weight in ["model","ema"]:
        model,n_params = load_model(ck,weight,DEV)
        assert n_params == 4948490
        with torch.inference_mode():
            plain = F.softmax(predict_logits(model,images,ck["norm"]["mean"],ck["norm"]["std"],DEV,EVAL_BATCH_SIZE),1).numpy()
            flipped = F.softmax(predict_logits(model,images,ck["norm"]["mean"],ck["norm"]["std"],DEV,EVAL_BATCH_SIZE,flip=True),1).numpy()
        for mode,probs in [("plain",plain),("flip",.5*(plain+flipped))]:
            ordered=probs[order]
            assert ordered.shape==(2000,10) and np.isfinite(ordered).all() and (ordered>=0).all() and np.allclose(ordered.sum(1),1,atol=1e-5)
            tag=f"{role}-ep{ck['epoch']}-{'raw' if weight=='model' else 'ema'}-{mode}"
            stem=f"{stamp}_{identity}_{tag}"
            csv_path=output_dir/(stem+"_submission.csv")
            probs_path=output_dir/(stem+"_probabilities.npz")
            report_path=output_dir/(stem+"_prediction-report.json")
            assert not any(p.exists() for p in [csv_path,probs_path,report_path])
            pd.DataFrame({"id":sample["id"],"label":ordered.argmax(1).astype(np.int64)}).to_csv(csv_path,index=False)
            validate_submission(csv_path,CACHE/"sample_submission.csv")
            np.savez_compressed(probs_path,ids=sample["id"].to_numpy(),probabilities=ordered)
            report={"run_id":ORIGINAL_RUN,"checkpoint_file":cp.name,"checkpoint_sha256":cp_hash,"checkpoint_role":role,"epoch":ck["epoch"],"weights":weight,"tta":mode=="flip","inference":"fp32 eager; original prediction functions; probability averaging for horizontal flip","n_params":n_params,"cache_fingerprint":META["fingerprint"],"model_fingerprint":ck["model_fingerprint"],"normalization":ck["norm"],"rows":2000,"format_check":"passed","submission":csv_path.name,"submission_sha256":sha256_file(csv_path),"probabilities_sha256":sha256_file(probs_path)}
            report_path.write_text(json.dumps(report,indent=2)+"\n")
            created.extend([csv_path,probs_path,report_path]);reports.append(report)
            print("EXPORTED",tag,csv_path.name,flush=True)
        del model;gc.collect();torch.cuda.empty_cache()
    assert sha256_file(cp)==cp_hash
assert len(reports)==8
bundle_manifest={"run_id":ORIGINAL_RUN,"checkpoint_files":2,"weight_states":4,"submissions":8,"checkpoint_bytes_in_bundle":False,"reports":reports,"files":[{"name":p.name,"bytes":p.stat().st_size,"sha256":sha256_file(p)} for p in created]}
manifest_path=output_dir/(f"{stamp}_{identity}_all-checkpoint-exports_manifest.json")
manifest_path.write_text(json.dumps(bundle_manifest,indent=2)+"\n")
bundle_path=output_dir/(f"{stamp}_{identity}_all-checkpoint-exports.zip")
with zipfile.ZipFile(bundle_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in created+[manifest_path]:z.write(p,p.name)
print("ALL_CHECKPOINT_EXPORTS_COMPLETE",bundle_path,"sha256",sha256_file(bundle_path),flush=True)


Saved checkpoint files: ['11-04-01-58_E_pyramid-ver-33-T128-asam-t122reg_best-ckpt.pt', '11-04-01-58_E_pyramid-ver-33-T128-asam-t122reg_last-ckpt.pt']
EXPORTED best-ep848-raw-plain 11-04-14-29_E_pyramid-ver-33-T128-asam-t122reg_best-ep848-raw-plain_submission.csv
EXPORTED best-ep848-raw-flip 11-04-14-29_E_pyramid-ver-33-T128-asam-t122reg_best-ep848-raw-flip_submission.csv
EXPORTED best-ep848-ema-plain 11-04-14-29_E_pyramid-ver-33-T128-asam-t122reg_best-ep848-ema-plain_submission.csv
EXPORTED best-ep848-ema-flip 11-04-14-29_E_pyramid-ver-33-T128-asam-t122reg_best-ep848-ema-flip_submission.csv
EXPORTED last-ep900-raw-plain 11-04-14-29_E_pyramid-ver-33-T128-asam-t122reg_last-ep900-raw-plain_submission.csv
EXPORTED last-ep900-raw-flip 11-04-14-29_E_pyramid-ver-33-T128-asam-t122reg_last-ep900-raw-flip_submission.csv
EXPORTED last-ep900-ema-plain 11-04-14-29_E_pyramid-ver-33-T128-asam-t122reg_last-ep900-ema-plain_submission.csv
EXPORTED last-ep900-ema-flip 11-04-14-29_E_pyramid-ver-33-T128-a